# TANGLE → PuMA cross-validation

This notebook builds one deterministic TANGLE configuration, runs exact centerline analysis in Rust, exports a VTI/JSON bundle in Rust, and then imports that bundle with `pumapy` directly. PuMA is optional and is not wrapped by TANGLE.

In [ ]:
from pathlib import Path
import json
import tangle

CELL_LENGTH = 100e-6
FIBER_LENGTH = 80e-6
FIBER_DIAMETER = 10e-6
AXIS_SEPARATION = 12e-6
VOXEL_SIZE = 2e-6
OUTPUT = Path('output/puma_cross_validation.puma')

## 1. One canonical centerline configuration

The fibers have equal length and diameter, point along x and y, and are separated by 12 µm. Their 5 µm radii therefore leave a 2 µm physical gap. The exact expected orientation tensor has `Axx = Ayy = 0.5`.

In [ ]:
center = CELL_LENGTH / 2
half_length = FIBER_LENGTH / 2
half_gap = AXIS_SEPARATION / 2
material = tangle.Material('validation fiber', diameter=FIBER_DIAMETER)
fibers = tangle.FiberCollection('orthogonal pair')
fibers.add_fiber([
    [center - half_length, center, center - half_gap],
    [center + half_length, center, center - half_gap],
], material)
fibers.add_fiber([
    [center, center - half_length, center + half_gap],
    [center, center + half_length, center + half_gap],
], material)
assembly = tangle.Assembly(tangle.Cell([CELL_LENGTH] * 3))
tangle.Recipe(assembly).insert(fibers, name='orthogonal pair')
assembly

## 2. Exact TANGLE analysis

`characterize()` calls `tangle_characterize` through PyO3. It uses the actual centerlines and sections—not a NumPy reconstruction and not the voxel image.

In [ ]:
analysis = assembly.characterize()
{
    'nominal_swept_volume_fraction': analysis.nominal_swept_volume_fraction,
    'length_weighted_orientation_tensor': analysis.length_weighted_orientation_tensor,
    'volume_weighted_orientation_tensor': analysis.volume_weighted_orientation_tensor,
    'segments': analysis.segment_count,
}

## 3. Rust voxelization and interoperable bundle

`export_puma()` calls `tangle_export::write_puma_bundle`. `domain.vti` contains cell-centered `phase_id` and `orientation`; the JSON files preserve definitions, ID maps, grid choices, and diagnostics.

In [ ]:
export = assembly.export_puma(OUTPUT, VOXEL_SIZE)
{
    'domain': export.domain_path,
    'voxel_counts': export.voxel_counts,
    'voxel_volume_fraction': export.voxel_volume_fraction,
    'ambiguous_voxels': export.ambiguous_voxels,
}

## 4. Independent PuMA analysis

This cell imports `pumapy` itself. If PuMA is absent, it explains the skipped step while leaving the TANGLE bundle usable. The comparison distinguishes TANGLE's nominal swept volume from the resolution-dependent occupied-voxel volume.

In [ ]:
try:
    import numpy as np
    import pumapy as puma
except ImportError:
    print('PuMA comparison skipped: install PuMA from conda-forge (conda create -n puma conda-forge::puma) and build TANGLE into that environment.')
else:
    workspace = puma.import_vti(str(export.domain_path), import_ws=True)
    last_phase = int(workspace.matrix.max())
    puma_vf = float(puma.compute_volume_fraction(workspace, (1, last_phase)))
    mask = (workspace.matrix >= 1) & (workspace.matrix <= last_phase)
    tangent = workspace.orientation[mask]
    voxel_tensor = np.einsum('ni,nj->ij', tangent, tangent) / tangent.shape[0]
    reference = np.asarray(analysis.volume_weighted_orientation_tensor)
    comparison = {
        'tangle_nominal_swept_volume_fraction': analysis.nominal_swept_volume_fraction,
        'puma_voxel_volume_fraction': puma_vf,
        'volume_fraction_difference': puma_vf - analysis.nominal_swept_volume_fraction,
        'orientation_frobenius_error': float(np.linalg.norm(voxel_tensor - reference)),
    }
    (OUTPUT / 'comparison.json').write_text(json.dumps(comparison, indent=2) + '\n')
    comparison

## Next: resolution convergence

Export this same `assembly` at several voxel sizes. Do not regenerate the fibers between resolutions. Plot volume-fraction and orientation errors against voxels per fiber diameter before applying expensive PuMA analyses to a felt crop.